In [26]:
import argparse
import math
import os

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from dataclasses import dataclass
from torchvision.utils import make_grid, save_image
from typing import Optional
import einx
from tasks.utils import sem_entropy, cosine_warmup_get_value

In [15]:
@dataclass
class HSEMHeadConfig:
    L: int
    V: int
    D: int
    temp: float
    input_dim: int
    per_simpex_ln: bool = False
    out_normalization: str = "none"


class HSEMHead(nn.Module):
    def __init__(self, cfg: Optional[HSEMHeadConfig] = None, **kwargs):
        super().__init__()

        if cfg == None:
            cfg = HSEMHeadConfig(**kwargs)

        assert cfg.input_dim is not None, "input_dim has to be set"
        self.N = (cfg.V**cfg.D) // (cfg.V - 1)
        self.proj_in = nn.Linear(cfg.input_dim, cfg.L * self.N * cfg.V, bias=False)
        if cfg.per_simpex_ln:
            self.norm = nn.LayerNorm(
                (cfg.V,),
            )
        else:
            self.norm = nn.LayerNorm((cfg.L, self.N, cfg.V))
        self.proj_out = nn.Linear(cfg.L * cfg.V * self.N, cfg.input_dim, bias=False)
        if cfg.out_normalization == "spectral":
            self.proj_out = nn.utils.parametrizations.spectral_norm(
                self.proj_out, eps=1e-6
            )
        elif cfg.out_normalization == "norm":
            self.proj_out = nn.utils.parametrizations.weight_norm(self.proj_out)
            self.proj_out.weight_g.data.fill_(1)
        self.cfg = cfg

    @property
    def dlc_len(self):
        return self.cfg.D * self.cfg.L

    def forward(
        self,
        x: torch.Tensor,
        noise_std: float = 0.0,
        tau: Optional[float] = None,
    ):
        bs = x.shape[0]
        temp = self.cfg.temp if tau is None else tau

        # Compute conditional probabilities
        x = self.proj_in(x)
        x = einx.rearrange(
            "b (L N V) -> b L N V",
            x,
            L=self.cfg.L,
            N=self.N,
            V=self.cfg.V,
        )
        x = self.norm(x)
        x = F.softmax(x / temp, -1)

        # Compute DLC probs (i.e. the joint) by going down tree
        # E.g. p(x_0,x_1,x_2) = p(x_0) * p(x_1 | x_0) * p(x_2 | x_0, x_1)
        with torch.autocast(device_type="cuda", dtype=torch.float32):
            parent_probs = torch.ones(
                size=(bs, self.cfg.L, 1), device=x.device, dtype=x.dtype
            )
            start = 0
            probs = []
            for d in range(self.cfg.D):
                # Compute probs at level d by multiplying with parent probs
                end = start + self.cfg.V**d
                level = x[:, :, start:end] * parent_probs[..., None]
                probs.append(level)
                # Update parent and go down a level
                parent_probs = einx.rearrange("b L n V -> b L (n V)", level)
                start = end

        # Compute output
        if noise_std > 0.0:
            out = torch.cat(
                [
                    p
                   # + (noise_std * 1/math.sqrt(p.shape[-1] * p.shape[-2]))
                    + noise_std
                    * torch.randn_like(p)
                    for p in probs
                ],
                dim=2,
            )
        else:
            out = torch.cat(probs, dim=2)
        out = einx.rearrange("b L N V -> b (L N V)", out)
        out = self.proj_out(out)

        return out, probs

    def hard_codes(self, probs):
        hard_probs = []
        for p in probs:
            hard_probs.append(
                torch.softmax(einx.rearrange("b L N V -> b L (N V)", p) / 1e-10, dim=-1)
            )

        return torch.cat(hard_probs, dim=2), None

    def _usage_count(self, probs):
        counts = torch.concatenate(
            [
                torch.stack(
                    [torch.sum(p.argmax(-1) == i, dim=0) for i in range(self.cfg.V)],
                    dim=-1,
                )
                for p in probs
            ],
            dim=1,
        )
        counts = einx.rearrange("L N V -> (L N V)", counts)
        return counts

In [16]:
class SEMHead(nn.Module):
    def __init__(self, in_dim, L, V, tau=1.0):
        super().__init__()
        self.L = L
        self.V = V
        self.tau = tau
        self.lin = nn.Linear(in_dim, L * V)
        self.norm = nn.LayerNorm(V)
        self.proj_out = nn.Linear(L * V, in_dim)

    def forward(self, h, tau=None, noise_std=0.0):
        tau = self.tau if tau is None else tau
        logits = self.lin(h).view(h.size(0), self.L, self.V)
        logits = self.norm(logits)  # [B, L, V]
        probs = F.softmax(logits / tau, dim=-1)                   # [B, L, V]

        if noise_std > 0.0:
            #z = probs + (noise_std * 1/self.V) * torch.randn_like(probs)
            z = probs + noise_std * torch.randn_like(probs)
        else:
            z = probs

        z = self.proj_out(z.view(h.size(0), -1))

        return z, probs

    def hard_codes(self, logits):
        """
        logits: [B, L, V]
        returns:
          z_hard: [B, L, V] one-hot
          idx: [B, L] indices
        """
        idx = logits.argmax(dim=-1)  # [B, L]
        z_hard = F.one_hot(idx, num_classes=self.V).float()
        return z_hard, idx


class SEMAutoencoder(nn.Module):
    def __init__(self, x_dim=28 * 28, hidden_dim=512, bottleneck_dim=256,
                 L=8, V=16, D=0, tau=1.0, big_decoder=False):
        super().__init__()
        self.x_dim = x_dim
        self.encoder = nn.Sequential(
            nn.Linear(x_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, bottleneck_dim),
            nn.ReLU(),
        )
        if D == 0:
            self.sem = SEMHead(bottleneck_dim, L, V, tau=tau)
        else:
            sem_cfg = HSEMHeadConfig(
                L=L,
                V=V,
                D=D,
                temp=tau,
                input_dim=bottleneck_dim,
                per_simpex_ln=True,
                out_normalization="none",
            )
            self.sem = HSEMHead(cfg=sem_cfg)
        if big_decoder:
            self.decoder = nn.Sequential(
                nn.ReLU(),
                nn.Linear(bottleneck_dim, hidden_dim*4),
                nn.ReLU(),
                nn.Linear(hidden_dim*4, hidden_dim*8),
                nn.ReLU(),
                nn.Linear(hidden_dim*8, hidden_dim*8),
                nn.ReLU(),
                nn.Linear(hidden_dim*8, hidden_dim*4),
                nn.ReLU(),
                nn.Linear(hidden_dim*4, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, x_dim),
                nn.Sigmoid(),  # MNIST pixels in [0,1]
            )
        else:
            self.decoder = nn.Sequential(
                nn.ReLU(),
                nn.Linear(bottleneck_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, x_dim),
                nn.Sigmoid(),  # MNIST pixels in [0,1]
            )

    def forward(self, x, tau=None, noise_std=0.0):
        # x: [B, x_dim]
        h = self.encoder(x)
        z, probs = self.sem(h, tau=tau, noise_std=noise_std)
        y = self.decoder(z)
        return y, z, probs
    
def visualize(model, test_loader, device, args):
    model.eval()
    x, _ = next(iter(test_loader))
    x = x.to(device)  # [B, 784]

    # Use final tau, no noise for visualization
    tau = args.tau_final
    noise_std = 0.0

    with torch.no_grad():
        # soft recon
        y_soft, z, logits = model(x, tau=tau, noise_std=noise_std)

        # hard recon via argmax codes
        z_hard, _ = model.sem.hard_codes(logits)        # [B, L, V]
        y_hard = model.decoder(model.sem.proj_out(z_hard.view(z_hard.size(0), -1)))

    # reshape to images
    B = min(args.n_vis, x.size(0))
    x_img = x[:B].view(B, 1, 28, 28)
    y_soft_img = y_soft[:B].view(B, 1, 28, 28)
    y_hard_img = y_hard[:B].view(B, 1, 28, 28)

    # stack: row1 = original, row2 = soft recon, row3 = hard recon
    grid = torch.cat([x_img, y_soft_img, y_hard_img], dim=0)  # [3B, 1, 28, 28]
    grid = make_grid(grid, nrow=B, padding=2)

    os.makedirs(args.out_dir, exist_ok=True)
    out_path = os.path.join(args.out_dir, f"vis_sem_ae_{args.mode}.png")
    save_image(grid, out_path)
    print(f"Saved visualization grid to {out_path}")

def evaluate_test_loss(model, test_loader, device, args):
    model.eval()
    total_soft = 0.0
    total_hard = 0.0
    n = 0

    with torch.no_grad():
        for x, _ in test_loader:
            x = x.to(device)
            B = x.size(0)

            # use final temperature and no noise
            tau = args.tau_final

            # soft reconstruction
            y_soft, z, logits = model(x, tau=tau, noise_std=0.0)
            if args.bce_loss:
                loss_soft = F.binary_cross_entropy(y_soft, x, reduction="sum").item()
            else:
                loss_soft = F.mse_loss(y_soft, x, reduction="sum").item()

            # hard reconstruction
            z_hard, _ = model.sem.hard_codes(logits)
            y_hard = model.decoder(model.sem.proj_out(z_hard.view(B, -1)))
            if args.bce_loss:
                loss_hard = F.binary_cross_entropy(y_hard, x, reduction="sum").item()
            else:
                loss_hard = F.mse_loss(y_hard, x, reduction="sum").item()

            total_soft += loss_soft
            total_hard += loss_hard
            n += B

    return total_soft / n, total_hard / n

In [17]:
@dataclass
class CFG:
    mode: str = 'noise'
    L: int = 8
    V: int = 16
    D: int = 0
    tau_init: float = 1.0
    tau_final: float = 0.3
    hidden_dim: int = 512
    bottleneck_dim: int = 256
    lambda_H_max: float = 1.0
    noise_std_max: float = 0.1
    epochs: int = 5
    batch_size: int = 256
    lr: float = 1e-3
    bce_loss: bool = True
    data_dir: str = "./toy/data"
    out_dir: str = "./toy/checkpoints"
    save_model: bool = False
    log_every: int = 200
    cpu: bool = False
    visualize: bool = True
    n_vis: int = 3
    big_decoder: bool = False

In [35]:
def train(args : CFG):
    codes = []

    device = "cuda" if torch.cuda.is_available() and not args.cpu else "cpu"
    print(f"Using device: {device}")

    # MNIST data
    transform = transforms.Compose([
        transforms.ToTensor(),     # [0,1]
        transforms.Lambda(lambda t: t.view(-1))  # flatten 28x28 -> 784
    ])
    train_ds = datasets.MNIST(
        root=args.data_dir, train=True, download=True, transform=transform
    )
    test_ds = datasets.MNIST(
        root=args.data_dir, train=False, download=True, transform=transform
    )

    train_loader = DataLoader(
        train_ds, batch_size=args.batch_size, shuffle=True, num_workers=4, pin_memory=True
    )
    test_loader = DataLoader(
        test_ds, batch_size=args.batch_size, shuffle=False, num_workers=4, pin_memory=True
    )

    model = SEMAutoencoder(
        x_dim=28 * 28,
        hidden_dim=args.hidden_dim,
        bottleneck_dim=args.bottleneck_dim,
        L=args.L, V=args.V, D=args.D,
        tau=args.tau_init,
        big_decoder=args.big_decoder
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=args.lr)

    # total steps for schedule
    steps_per_epoch = math.ceil(len(train_loader) / 1.0)
    total_steps = args.epochs * steps_per_epoch

    global_step = 0

    for epoch in range(args.epochs):
        model.train()
        for x, _ in train_loader:
            x = x.to(device)

            # fraction of training completed
            frac = min(global_step / max(1, total_steps), 1.0)

            # schedules
            if args.mode in ["entropy", "both"]:
                lambda_H = cosine_warmup_get_value(global_step, args.lambda_H_max, 0)
            else:
                lambda_H = 0.0

            if args.mode in ["noise", "both"]:
                noise_std =  cosine_warmup_get_value(global_step, args.noise_std_max, 1500, 1)
            else:
                noise_std = 0.0

            # temperature annealing (optional)
            tau = args.tau_init - frac * (args.tau_init - args.tau_final)
            tau = max(tau, args.tau_final)

            # forward
            y, z, probs = model(x, tau=tau, noise_std=noise_std)

            # reconstruction loss (MSE or BCE)
            if args.bce_loss:
                recon_loss = F.binary_cross_entropy(y, x, reduction="mean")
            else:
                recon_loss = F.mse_loss(y, x, reduction="mean")

            H_sample, H_marg = sem_entropy(probs)

            loss = recon_loss + lambda_H * (H_sample - H_marg)

            opt.zero_grad()
            loss.backward()
            opt.step()

            if global_step % args.log_every == 0:
                with torch.no_grad():
                    

                    # hard reconstruction (argmax codes)
                    z_hard, _ = model.sem.hard_codes(probs)
                
                    y_hard = model.decoder(model.sem.proj_out(z_hard.view(z_hard.size(0), -1)))
                    if args.bce_loss:
                        hard_recon = F.binary_cross_entropy(y_hard, x, reduction="mean").item()
                    else:
                        hard_recon = F.mse_loss(y_hard, x, reduction="mean").item()

                # evaluate on test set
                test_soft, test_hard = evaluate_test_loss(model, test_loader, device, args)
                
                print(
                    f"[epoch {epoch:02d} step {global_step:06d}] "
                    f"mode={args.mode} "
                    f"loss={loss.item():.4f} "
                    f"val_loss={test_soft:.4f} "
                    f"recon={recon_loss.item():.4f} "
                    f"recon_hard={hard_recon:.4f} "
                    f"H_sample={H_sample.item():.3f} "
                    f"H_marg={H_marg:.3f} "
                    f"tau={tau:.3f} "
                    f"lambda_H={lambda_H:.4f} "
                    f"noise={noise_std:.3f}"
                )

            global_step += 1

    # save model if asked
    os.makedirs(args.out_dir, exist_ok=True)
    if args.save_model:
        torch.save(model.state_dict(), os.path.join(args.out_dir, f"sem_ae_{args.mode}.pt"))
        print("Model saved.")

    # visualization step
    if args.visualize:
        visualize(model, test_loader, device, args)
    return codes

In [ ]:
xd = train(
    CFG(
        tau_final=1.0,
        L=8,
        V=8,
        D=4,
        epochs=10,
        lambda_H_max=0.05,
        noise_std_max=0.3,
        mode="noise",
        big_decoder=True,
    )
)

Using device: cuda
[epoch 00 step 000000] mode=noise loss=0.6934 val_loss=537.3925 recon=0.6934 recon_hard=0.6854 H_sample=0.810 H_marg=0.886 tau=1.000 lambda_H=0.0000 noise=0.000
[epoch 00 step 000200] mode=noise loss=0.2318 val_loss=183.9746 recon=0.2318 recon_hard=0.2389 H_sample=0.690 H_marg=0.759 tau=1.000 lambda_H=0.0000 noise=0.013
[epoch 01 step 000400] mode=noise loss=0.1985 val_loss=159.4412 recon=0.1985 recon_hard=0.2018 H_sample=0.498 H_marg=0.689 tau=1.000 lambda_H=0.0000 noise=0.050
[epoch 02 step 000600] mode=noise loss=0.1869 val_loss=139.0603 recon=0.1869 recon_hard=0.1872 H_sample=0.358 H_marg=0.589 tau=1.000 lambda_H=0.0000 noise=0.104
[epoch 03 step 000800] mode=noise loss=0.1725 val_loss=129.1266 recon=0.1725 recon_hard=0.1694 H_sample=0.254 H_marg=0.518 tau=1.000 lambda_H=0.0000 noise=0.166
[epoch 04 step 001000] mode=noise loss=0.1741 val_loss=126.1399 recon=0.1741 recon_hard=0.1691 H_sample=0.201 H_marg=0.484 tau=1.000 lambda_H=0.0000 noise=0.225
[epoch 05 step 

In [19]:
1.0/math.sqrt(8)

0.35355339059327373

In [ ]:
0.5,0.5